# 🏃 Boston Marathon — Exploratory Data Analysis (EDA)

**Dataset**: Boston Marathon Winners (Men's 1897–2024, Women's 1966–2024)

**Objective**: Load, clean, explore, and analyze the marathon data to derive meaningful insights.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('dark_background')
sns.set_palette(['#a855f7', '#3b82f6', '#06b6d4', '#f472b6', '#34d399', '#fbbf24'])
print('Libraries loaded successfully!')

## 2. Load the Dataset

In [ ]:
# Load both datasets
mens = pd.read_csv('../data/Mens_Boston_Marathon_Winners_r0l7bV.csv')
womens = pd.read_csv('../data/Womens_Boston_Marathon_Winners_8SSnWb.csv')

print(f"Men's dataset shape: {mens.shape}")
print(f"Women's dataset shape: {womens.shape}")
print(f"\nMen's columns: {list(mens.columns)}")
print(f"Women's columns: {list(womens.columns)}")

In [ ]:
# Preview Men's data
print("=== MEN'S DATA (First 10 rows) ===")
mens.head(10)

In [ ]:
# Preview Women's data
print("=== WOMEN'S DATA (First 10 rows) ===")
womens.head(10)

## 3. Data Cleaning & Preprocessing

In [ ]:
# Add Gender column and merge
mens['Gender'] = 'Male'
womens['Gender'] = 'Female'
df = pd.concat([mens, womens], ignore_index=True)
print(f"Merged dataset shape: {df.shape}")
df.head()

In [ ]:
# Check data types
print("Data Types:")
print(df.dtypes)
print(f"\nNull values:\n{df.isnull().sum()}")
print(f"\nDuplicate rows: {df.duplicated().sum()}")

In [ ]:
# Convert Time to minutes
def time_to_minutes(t):
    try:
        parts = str(t).split(':')
        if len(parts) == 3:
            h, m, s = parts
            return int(h) * 60 + int(m) + int(s) / 60
    except:
        return np.nan
    return np.nan

df['Time_Minutes'] = df['Time'].apply(time_to_minutes)
df['Decade'] = (df['Year'] // 10) * 10
df['Decade_Label'] = df['Decade'].astype(str) + 's'
df['Pace_Per_Mile'] = df['Time_Minutes'] / df['Distance (Miles)']
df['Speed_MPH'] = df['Distance (Miles)'] / (df['Time_Minutes'] / 60)

print("Feature engineering complete!")
print(f"\nNew columns: Time_Minutes, Decade, Decade_Label, Pace_Per_Mile, Speed_MPH")
df.head()

In [ ]:
# Statistical summary
print("=== STATISTICAL SUMMARY ===")
df.describe()

In [ ]:
# Summary by Gender
print("=== SUMMARY BY GENDER ===")
df.groupby('Gender')[['Time_Minutes', 'Speed_MPH', 'Pace_Per_Mile']].agg(['mean', 'min', 'max', 'std'])

## 4. Exploratory Visualizations

In [ ]:
# 4.1 Distribution of Finishing Times
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df[df['Gender']=='Male']['Time_Minutes'].dropna(), bins=20, 
             color='#3b82f6', alpha=0.8, edgecolor='black', label='Male')
axes[0].hist(df[df['Gender']=='Female']['Time_Minutes'].dropna(), bins=15, 
             color='#f472b6', alpha=0.7, edgecolor='black', label='Female')
axes[0].set_title('Distribution of Finishing Times', fontweight='bold')
axes[0].set_xlabel('Time (Minutes)')
axes[0].set_ylabel('Frequency')
axes[0].legend()

sns.boxplot(data=df, x='Gender', y='Time_Minutes', 
            palette={'Male': '#3b82f6', 'Female': '#f472b6'}, ax=axes[1])
axes[1].set_title('Time Distribution by Gender', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 4.2 Winning Times Trend Over Years
fig, ax = plt.subplots(figsize=(14, 6))

for gender, color in [('Male', '#3b82f6'), ('Female', '#f472b6')]:
    subset = df[df['Gender'] == gender].sort_values('Year')
    ax.plot(subset['Year'], subset['Time_Minutes'], 'o-', color=color, 
            markersize=4, linewidth=1.5, alpha=0.8, label=gender)

ax.set_title('Boston Marathon Winning Times Over the Years', fontsize=14, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Time (Minutes)')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
# 4.3 Top Countries by Wins
fig, ax = plt.subplots(figsize=(10, 6))
top = df['Country'].value_counts().head(10)
colors = ['#a855f7', '#7c3aed', '#3b82f6', '#06b6d4', '#f472b6',
          '#34d399', '#fbbf24', '#f97316', '#ef4444', '#8b5cf6']
ax.barh(top.index[::-1], top.values[::-1], color=colors[:len(top)][::-1], 
        edgecolor='black', linewidth=0.5)
ax.set_title('Top 10 Countries by Total Wins', fontsize=14, fontweight='bold')
ax.set_xlabel('Number of Wins')
for i, v in enumerate(top.values[::-1]):
    ax.text(v + 0.3, i, str(v), va='center', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 4.4 Pie Chart — Country Distribution
fig, ax = plt.subplots(figsize=(8, 6))
country_counts = df['Country'].value_counts()
top_pie = country_counts.head(8)
if len(country_counts) > 8:
    top_pie['Others'] = country_counts.iloc[8:].sum()

wedges, texts, autotexts = ax.pie(top_pie.values, labels=top_pie.index, 
    autopct='%1.1f%%', colors=colors[:len(top_pie)], startangle=140)
ax.set_title('Winners by Country', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 4.5 Correlation Heatmap
fig, ax = plt.subplots(figsize=(8, 6))
numeric_cols = ['Year', 'Time_Minutes', 'Distance (Miles)', 'Distance (KM)', 'Pace_Per_Mile', 'Speed_MPH']
corr = df[numeric_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, annot=True, cmap='cool', fmt='.2f', ax=ax,
            linewidths=1, cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 4.6 Violin Plot
fig, ax = plt.subplots(figsize=(8, 5))
sns.violinplot(data=df, x='Gender', y='Time_Minutes', 
               palette={'Male': '#3b82f6', 'Female': '#f472b6'}, inner='quart', ax=ax)
ax.set_title('Finishing Time Distribution (Violin Plot)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 4.7 Count Plot by Decade
fig, ax = plt.subplots(figsize=(12, 5))
sns.countplot(data=df, x='Decade_Label', hue='Gender',
              palette={'Male': '#3b82f6', 'Female': '#f472b6'}, ax=ax,
              order=sorted(df['Decade_Label'].unique()))
ax.set_title('Number of Winners per Decade', fontsize=14, fontweight='bold')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# 4.8 Scatter Plot with Trend Line
fig, ax = plt.subplots(figsize=(12, 5))
for gender, color in [('Male', '#3b82f6'), ('Female', '#f472b6')]:
    subset = df[df['Gender'] == gender]
    ax.scatter(subset['Year'], subset['Time_Minutes'], c=color, s=40, 
               alpha=0.7, label=gender, edgecolors='white', linewidths=0.3)
    # Trend line
    z = np.polyfit(subset['Year'], subset['Time_Minutes'], 2)
    p = np.poly1d(z)
    x_line = np.linspace(subset['Year'].min(), subset['Year'].max(), 100)
    ax.plot(x_line, p(x_line), '--', color=color, alpha=0.5, linewidth=1.5)

ax.set_title('Year vs Finishing Time with Trend Lines', fontsize=14, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Time (Minutes)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 4.9 Area Chart — Cumulative Wins
fig, ax = plt.subplots(figsize=(12, 5))
for gender, color in [('Male', '#3b82f6'), ('Female', '#f472b6')]:
    subset = df[df['Gender'] == gender].sort_values('Year').copy()
    subset['Cumulative'] = range(1, len(subset) + 1)
    ax.fill_between(subset['Year'], subset['Cumulative'], alpha=0.3, color=color)
    ax.plot(subset['Year'], subset['Cumulative'], color=color, linewidth=2, label=gender)

ax.set_title('Cumulative Wins Over Time', fontsize=14, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Cumulative Wins')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Key Insights

1. **US Dominance in Early Years**: The United States dominated the Boston Marathon in the early decades.
2. **East African Rise**: Kenya and Ethiopia became dominant from the 1990s onward.
3. **Improving Times**: A clear downward trend in winning times shows improved athletic performance.
4. **Gender Gap**: Women's winning times have been progressively getting closer to men's.
5. **Distance Changes**: Marathon distance was standardized to 26.2 miles — early races were shorter.
6. **Speed Increase**: Average finishing speed has increased significantly over the decades.

In [ ]:
# Final summary statistics
print("=" * 50)
print("FINAL SUMMARY")
print("=" * 50)
print(f"Total records: {len(df)}")
print(f"Male records: {len(df[df['Gender']=='Male'])}")
print(f"Female records: {len(df[df['Gender']=='Female'])}")
print(f"Year range: {df['Year'].min()} - {df['Year'].max()}")
print(f"Countries represented: {df['Country'].nunique()}")
print(f"Fastest time (Men): {df[df['Gender']=='Male']['Time'].iloc[df[df['Gender']=='Male']['Time_Minutes'].idxmin()]}")
print(f"Fastest time (Women): {df[df['Gender']=='Female']['Time'].iloc[df[df['Gender']=='Female']['Time_Minutes'].idxmin()]}")
print(f"Average speed: {df['Speed_MPH'].mean():.2f} mph")